## Language Modeling

In [ ]:

import torch
import numpy as np
import evaluate
from accelerate import Accelerator
from transformers import EarlyStoppingCallback
from datasets import load_dataset
from transformers import pipeline
from transformers import DataCollatorWithPadding
from transformers import TrainingArguments, Trainer, set_seed
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, AutoModelForCausalLM
from transformers import DataCollatorForSeq2Seq
from transformers import set_seed
from sentence_transformers import SentenceTransformer
import os
import faiss
os.environ["WANDB_DISABLED"] = "true"
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import (
    set_seed,
    GPT2Config,
    GPT2Tokenizer,
    GPT2ForSequenceClassification,
    AdamW,
    get_scheduler,
    Adafactor,
    GPT2LMHeadModel,
    Adafactor,
)
import seaborn as sns




## Dataset Selection


In [ ]:
# Load the dataset
dataset = load_dataset("legacy-datasets/banking77")

# Inspect the dataset
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})


## About the Dataset Selected

The Banking77 dataset is a text classification dataset containing user queries from online banking platforms, with 77 distinct intent labels that correspond to various customer service topics such as "card activation," "loan inquiry," and "account balance." The dataset consists of 10,003 examples in the training set and 3,080 examples in the test set, with a default train-test split. This task is particularly useful for applications like customer support chatbots, intelligent assistants, or automated query routing in banking systems, where accurate intent classification enables timely and contextually appropriate responses to user queries.The inputs to the task are user queries, which are typically short pieces of text written in natural language, often containing domain-specific terms related to banking and financial services. The outputs, or labels, are one of 77 predefined intent categories that represent the core intents expressed in the queries. Examples of these labels include *card delivery*, *account balance inquiry*, *fraud report*, and *loan information*. This task presents several challenges, including the issue of overlapping intents, where some queries may fit into more than one category, leading to classification ambiguities. For instance, "How do I activate my new card?" could relate to both *card activation* and *account setup*. Additionally, the short length of user queries often results in a lack of sufficient context, making accurate classification more difficult. Another challenge is class imbalance, where certain intents may be overrepresented in the dataset, potentially causing the model to perform poorly on underrepresented classes. Furthermore, the dataset contains domain-specific terminology that may require models to have banking-related knowledge for accurate predictions. Lastly, noisy data, such as queries with typos, abbreviations, or non-standard grammar, can complicate preprocessing and understanding. Performance on this task is evaluated using metrics such as accuracy, which measures the percentage of correctly classified queries, and F1 score, which provides a balanced view of precision and recall. The use of a confusion matrix further helps analyze class-wise performance, identifying specific intents where the model struggles.

## Current Methods Used in this Dataset

The Banking77 dataset has been utilized in various studies to benchmark the performance of models on intent classification tasks. One of the state-of-the-art (SOTA) methods for this dataset leverages pre-trained transformer-based models, such as BERT (Bidirectional Encoder Representations from Transformers) and its derivatives (e.g., RoBERTa or DistilBERT). These models excel at text classification tasks due to their ability to capture contextual representations of words and phrases in a sentence. For instance, BERT-based models fine-tuned on the Banking77 dataset typically use a classification head added to the transformer architecture. This head consists of a fully connected layer followed by a softmax layer, which outputs probabilities for each of the 77 intents. Fine-tuning involves training the model on the dataset for a few epochs using a learning rate in the range of 2e-5 to 5e-5, with regularization techniques such as weight decay and dropout to prevent overfitting. A commonly used optimizer is AdamW, and metrics like accuracy and F1 score are used for evaluation. Models like RoBERTa, which modify BERT's pre-training objectives and use more extensive pre-training data, have also been shown to perform well on this task. In a notable benchmark study, RoBERTa-large achieved an accuracy of approximately 93% on the test set of the Banking77 dataset, demonstrating its effectiveness in capturing the nuances of user queries. Another method, fine-tuning lightweight models like DistilBERT, offers competitive performance (~90% accuracy) with reduced computational overhead, making it suitable for deployment in resource-constrained environments.Emerging approaches, such as zero-shot or few-shot learning using models like T5 or GPT-3, also show promise by leveraging transfer learning to classify intents without extensive labeled data. These models rely on pre-trained knowledge and perform inference based on task-specific prompts.

Sources:
Casanueva, I., Temcik, A., Gerz, D., Heinzerling, B., & Ruder, S. (2020). Efficient Intent Detection with Dual Sentence Encoders. arXiv preprint. https://arxiv.org/abs/2003.04807

Hugging Face Model Hub. Banking77 Results. https://huggingface.co/datasets/banking77

## Data Preprocessing

In [ ]:

# Parameters
set_seed(123)
batch_size = 16
epochs = 3
max_length = 60
device = torch.device("cuda")

train_val_split = dataset["train"].train_test_split(test_size=0.2, seed=123)
train_data = train_val_split["train"]
val_data = train_val_split["test"]
test_data = dataset["test"]

labels = train_data.features["label"].names  # Extract label names
print(f"LABELS: {labels}")

n_labels = len(labels)

print(f"Train size: {len(train_data)} | Val size: {len(val_data)} | Test size: {len(test_data)}")

LABELS: ['activate_my_card', 'age_limit', 'apple_pay_or_google_pay', 'atm_support', 'automatic_top_up', 'balance_not_updated_after_bank_transfer', 'balance_not_updated_after_cheque_or_cash_deposit', 'beneficiary_not_allowed', 'cancel_transfer', 'card_about_to_expire', 'card_acceptance', 'card_arrival', 'card_delivery_estimate', 'card_linking', 'card_not_working', 'card_payment_fee_charged', 'card_payment_not_recognised', 'card_payment_wrong_exchange_rate', 'card_swallowed', 'cash_withdrawal_charge', 'cash_withdrawal_not_recognised', 'change_pin', 'compromised_card', 'contactless_not_working', 'country_support', 'declined_card_payment', 'declined_cash_withdrawal', 'declined_transfer', 'direct_debit_payment_not_recognised', 'disposable_card_limits', 'edit_personal_details', 'exchange_charge', 'exchange_rate', 'exchange_via_app', 'extra_charge_on_statement', 'failed_transfer', 'fiat_currency_support', 'get_disposable_virtual_card', 'get_physical_card', 'getting_spare_card', 'getting_virtu

## Encoder only models


The chosen model is bert-base-uncased, a case-insensitive variant of BERT (Bidirectional Encoder Representations from Transformers) introduced by Devlin et al. in 2018. BERT is an encoder-only transformer model that uses 12 transformer encoder layers, each with 768 hidden dimensions and 12 self-attention heads, totaling approximately 110 million parameters. Its training involves two objectives: Masked Language Modeling (MLM), which predicts randomly masked words to enable bidirectional contextual understanding, and Next Sentence Prediction (NSP), which determines sentence relationships. These features allow BERT to excel in tasks requiring contextual comprehension, such as text classification, question answering, and sentence pair analysis. The key advantage of BERT lies in its bidirectional architecture, which considers both left and right contexts during training, offering a more nuanced understanding of language compared to unidirectional models like GPT. Additionally, BERT's pretraining on a large corpus provides a strong foundation for fine-tuning with minimal task-specific modifications, often requiring only the addition of a simple classification layer. BERT has achieved state-of-the-art results across multiple NLP benchmarks, such as SQuAD and GLUE, and its WordPiece tokenizer ensures efficient handling of rare words by splitting them into subword units. However, BERT has limitations, including its large size and bidirectional nature, making it memory and computation-intensive, requiring significant resources for training and fine-tuning. Furthermore, as an encoder-only model, BERT is tailored for understanding tasks like classification and sentiment analysis but lacks generative capabilities, making it unsuitable for tasks such as text generation. Additionally, its Next Sentence Prediction (NSP) objective has been criticized for being less impactful in some applications, leading to its exclusion in models like RoBERTa.

Sources:

Devlin, J., Chang, M.-W., Lee, K., & Toutanova, K. (2019). BERT: Pre-training of deep bidirectional transformers for language understanding. arXiv preprint arXiv:1810.04805. Retrieved from https://arxiv.org/abs/1810.04805

Xiao, H. (2019, March 4). BERT explained: State of the art natural language processing for NLP beginners. Medium. Retrieved from
https://medium.com/@dhartidhami/understanding-bert-word-embeddings-7dc4d2ea54ca

## Finetuning the Model on the Banking77 Dataset and Evaluating Performance

In [ ]:
# Load tokenizer and model
encoder_model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(encoder_model_checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(encoder_model_checkpoint, num_labels=77)

# Tokenize datasets
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_train = train_data.map(tokenize_function, batched=True)
tokenized_val = val_data.map(tokenize_function, batched=True)
tokenized_test = test_data.map(tokenize_function, batched=True)

# Data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

# Compute metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    accuracy = accuracy_score(labels, predictions)
    return {"accuracy": accuracy}

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Train the model
print("Starting training...")
trainer.train()

# Evaluate on the test set
print("Evaluating on the test set...")
metrics = trainer.evaluate(tokenized_test)

# Simplified metrics output
simplified_metrics = {
    "loss": metrics["eval_loss"],
    "learning_rate": training_args.learning_rate,
    "epoch": training_args.num_train_epochs,
    "accuracy": metrics["eval_accuracy"],
}
print(simplified_metrics)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/2001 [00:00<?, ? examples/s]

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
<ipython-input-49-3464b63d01de>:41: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting training...


Epoch,Training Loss,Validation Loss,Accuracy
1,1.752700,1.634199,0.765117
2,0.822400,0.832173,0.863068
3,0.572400,0.664310,0.884558


Evaluating on the test set...


{'loss': 0.6913717985153198, 'learning_rate': 2e-05, 'epoch': 3, 'accuracy': 0.8779220779220779}


## Decoder only models

GPT-2 (Generative Pre-trained Transformer 2) is a decoder-only transformer model developed by OpenAI. It is designed for unidirectional language modeling, processing input text sequentially to predict the next word based on the context provided by preceding words. Pre-trained on a vast corpus of internet text, GPT-2 learns rich language representations and can be fine-tuned for downstream tasks such as text classification, summarization, and question answering. It uses a unidirectional architecture, meaning it processes text from left to right, making it particularly well-suited for tasks that rely on sequential dependencies. One of GPT-2's unique properties is its ability to adapt flexibly to a wide range of tasks through fine-tuning. This is achieved thanks to its robust pre-training on a diverse dataset, which ensures a strong foundational understanding of language. Additionally, the model employs sinusoidal positional encodings to preserve word order within a sequence, and it is available in multiple configurations (e.g., 125M, 355M, 774M, and 1.5B parameters), making it adaptable to varying computational resource constraints. One of GPT-2's advantages are its strong language generation capabilities and task-agnostic pre-training, which enables it to generalize effectively across different NLP tasks, as well as efficient fine-tuning, often requiring relatively small datasets to achieve competitive performance. However, the model has some limitations. Its unidirectional nature restricts it from understanding bidirectional context, which can impact its ability to capture the full semantics of a sentence. Additionally, its large size and high computational cost can be barriers for fine-tuning and deployment, particularly on resource-constrained devices. GPT-2 is also prone to overfitting when fine-tuned on small or domain-specific datasets and may inherit biases from its pre-training corpus, which raises ethical concerns.

Sources:

Hugging Face. (n.d.). GPT-2 Model Card. Retrieved from https://huggingface.co/openai-community/gpt2

Radford, A., et al. (2019). "GPT-2: Better Language Models and Their Implications." OpenAI Blog. Available at: https://openai.com/blog/better-language-models

## Loading the Decoder only Model and Reporting Performance

In [ ]:
model_name = "gpt2"
torch.manual_seed(123)
num_epochs = 3
seq_max_length = 60
model_name = "gpt2"
num_classes = len(train_data.features["label"].names)  # Total number of labels

# Prompt Template
def format_prompt(text):
    return f"Determine the intent of the following text:\n{text}\nReply with the intent label enclosed in <label_start> and <label_end>."

# Dataset Class
class IntentDataset(Dataset):
    def __init__(self, data, tokenizer, max_length):
        self.texts = data["text"]
        self.labels = data["label"]
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        prompt = format_prompt(text)
        encoded = self.tokenizer(
            prompt,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": encoded["input_ids"].squeeze(),
            "attention_mask": encoded["attention_mask"].squeeze(),
            "label": torch.tensor(label, dtype=torch.long),
        }


# GPT-2 Classification Collator
class ClassificationCollator:
    def __init__(self, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __call__(self, batch):
        input_ids = torch.stack([item["input_ids"] for item in batch])
        attention_mask = torch.stack([item["attention_mask"] for item in batch])
        labels = torch.stack([item["label"] for item in batch])
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }


# Model and Tokenizer Setup
print("Loading GPT-2 configuration...")
config = GPT2Config.from_pretrained(model_name, num_labels=num_classes)

print("Loading GPT-2 tokenizer...")
gpt2_tokenizer = GPT2Tokenizer.from_pretrained(model_name)
gpt2_tokenizer.padding_side = "left"
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token  # Set padding token to EOS

print("Loading GPT-2 model...")
classification_model = GPT2ForSequenceClassification.from_pretrained(model_name, config=config)
classification_model.resize_token_embeddings(len(gpt2_tokenizer))
classification_model.config.pad_token_id = classification_model.config.eos_token_id
classification_model.to(device)


# Datasets and Dataloaders
print("Preparing datasets...")
train_dataset = IntentDataset(train_data, gpt2_tokenizer, seq_max_length)
test_dataset = IntentDataset(test_data, gpt2_tokenizer, seq_max_length)

collator = ClassificationCollator(gpt2_tokenizer, seq_max_length)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collator)
test_loader = DataLoader(test_dataset, batch_size=batch_size, collate_fn=collator)


# Optimizer and Scheduler
optimizer = Adafactor(
    classification_model.parameters(),
    lr=1e-3,
    scale_parameter=False,
    relative_step=False,
    warmup_init=False,
)


# Training Function
def train_model(dataloader, model, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for batch in tqdm(dataloader, desc="Training"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        logits = outputs.logits

        # Backpropagation
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        # Track predictions and labels
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=-1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(batch["labels"].cpu().tolist())

    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    return avg_loss, accuracy


# Validation Function
def evaluate_model(dataloader, model, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            logits = outputs.logits

            # Track predictions and labels
            total_loss += loss.item()
            preds = torch.argmax(logits, dim=-1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(batch["labels"].cpu().tolist())

    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    return avg_loss, accuracy, all_labels, all_preds


# Training Loop
print("Starting training...")
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")
    train_loss, train_acc = train_model(train_loader, classification_model, optimizer, lr_scheduler, device)
    print(f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_acc:.4f}")

    val_loss, val_acc, _, _ = evaluate_model(test_loader, classification_model, device)
    print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc:.4f}")


# Final Evaluation on Test Set
print("Evaluating on test set...")
test_loss, test_acc, true_labels, predictions = evaluate_model(test_loader, classification_model, device)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")


Loading GPT-2 configuration...
Loading GPT-2 tokenizer...
Loading GPT-2 model...


Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Preparing datasets...
Starting training...
Epoch 1/3


Training:   0%|          | 0/501 [00:00<?, ?it/s]

Train Loss: 3.6694, Train Accuracy: 0.1022


...

Train Loss: 0.7550, Train Accuracy: 0.7663


Evaluating:   0%|          | 0/193 [00:00<?, ?it/s]

Validation Loss: 0.7908, Validation Accuracy: 0.7731
Evaluating on test set...


Evaluating:   0%|          | 0/193 [00:00<?, ?it/s]

Test Loss: 0.7908, Test Accuracy: 0.7731


Error Analysis


Encoder performance:

The encoder-only model demonstrates strong overall performance, achieving an accuracy of 88%. It excels in categories with distinct textual patterns, such as age_limit, apple_pay_or_google_pay, and activate_my_card, where it achieves near-perfect precision and recall. This indicates that the model effectively captures textual features in well-structured contexts. However, it struggles with ambiguous or overlapping categories, such as virtual_card_not_working and contactless_not_working, where precision and recall drop significantly. For example, it incorrectly predicts getting_virtual_card for the input "My virtual card won't work," and misclassifies contactless_not_working as lost_or_stolen_phone for the input "Fix my contactless". These errors suggest that the model struggles to distinguish between functionally similar categories when textual cues are vague. On the other hand, the encoder-only model performs well for contextually unique inputs, such as correctly classifying pending_top_up for the input "Why hasn't my top-up been completed?" This highlights its ability to leverage clear signals for precise predictions.

Decoder performance:

The decoder-only model, with an accuracy of 77%, lags behind the encoder-only model in overall performance. It shows decent results for categories like card_payment_not_recognised and why_verify_identity, where textual patterns align closely with the training data. For instance, it correctly predicts card_payment_not_recognised for the input "I've been unable to pay contactless with my card for over a week now." However, the model is prone to misclassifying broader or more abstract categories like top_up_failed, transfer_timing, and extra_charge_on_statement. Notable errors include the misclassification of top_up_failed as top_up_reverted for the input "Where's the money that got charged to my card? It's not showing up in my account balance," and the prediction of card_linking instead of lost_or_stolen_card for the input "Do you support any currency?" These errors suggest that the decoder-only model struggles with nuanced understanding or indirect phrasing. Nevertheless, it performs well on simpler inputs, such as correctly predicting balance_not_updated_after_bank_transfer for the input "How can someone add money to my account?".

Strengths and weaknesses:

Both models show distinct strengths and weaknesses. The encoder-only model is highly effective in classifying tasks with clear and distinct textual cues, leveraging its bidirectional structure to excel in structured and context-dependent queries. The decoder-only model, while less accurate overall, performs well on queries with straightforward phrasing and aligns well with its generation-focused architecture. However, both models struggle with overlapping or ambiguous categories, where textual patterns are less defined, with these challenges being more pronounced in the decoder-only model. Additionally, poor performance in certain categories, such as contactless_not_working and virtual_card_not_working, reflects insufficient generalization to edge cases.

Suggested changes:

 Encoder errors often stem from overlapping semantic content, and addressing these requires balanced paraphrasing and augmenting data for underrepresented categories. Additionally, fine-tuning with hierarchical tasks like paraphrase identification can improve its understanding of nuanced differences. On the other hand, the decoder-only model, with an accuracy of 77%, struggles with precision in complex contexts, such as misclassifying "top_up_failed" as "top_up_reverted" or handling diverse prompts effectively. To address this, prompt engineering can improve contextual comprehension, and data augmentation can help mitigate biases and class imbalances.Upgrading the optimizer to Adam, and fine-tuning the model with the transformer trainer class could produce better results as well.

In [ ]:
import random


model_name = "gpt2"

def error_analysis(trainer, tokenized_test, original_test_data, label_names, model_name):
    """
    Perform error analysis by comparing model predictions and ground truth.
    """
    predictions = trainer.predict(tokenized_test)
    predicted_labels = np.argmax(predictions.predictions, axis=1)
    true_labels = predictions.label_ids

    print(f"Classification Report:")
    print(classification_report(true_labels, predicted_labels, target_names=label_names, zero_division=0))

    errors = [
        {
            "text": original_test_data[i]["text"],  # Access text from the original dataset
            "true": label_names[true_labels[i]],
            "predicted": label_names[predicted_labels[i]],
        }
        for i in range(len(true_labels)) if true_labels[i] != predicted_labels[i]
    ]

    correct = [
        {
            "text": original_test_data[i]["text"],  # Access text from the original dataset
            "true": label_names[true_labels[i]],
            "predicted": label_names[predicted_labels[i]],
        }
        for i in range(len(true_labels)) if true_labels[i] == predicted_labels[i]
    ]

    # Randomly sample 5 examples
    random_errors = random.sample(errors, min(len(errors), 5)) if errors else []
    random_correct = random.sample(correct, min(len(correct), 5)) if correct else []

    print(f"\nRandom Sample of Incorrect Predictions for {model_name}:")
    for error in random_errors:
        print(error)

    print(f"\nRandom Sample of Correct Predictions for {model_name}:")
    for correct_example in random_correct:
        print(correct_example)

print("Error Analysis for Encoder-Only Model:")
error_analysis(trainer, tokenized_test, test_data, train_data.features["label"].names, "Encoder-Only Model")

def evaluate_with_examples(dataloader, model, device, test_data, label_names):
    """
    Evaluate the decoder model and extract examples of correct and incorrect predictions.
    """
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    correct_examples, incorrect_examples = [], []

    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(dataloader, desc="Evaluating")):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            logits = outputs.logits

            # Track predictions and labels
            total_loss += loss.item()
            preds = torch.argmax(logits, dim=-1).cpu().tolist()
            labels = batch["labels"].cpu().tolist()

            all_preds.extend(preds)
            all_labels.extend(labels)

            # Collect examples
            for i, (pred, true_label) in enumerate(zip(preds, labels)):
                text = test_data[batch_idx * batch_size + i]["text"]
                example = {
                    "text": text,
                    "true": label_names[true_label],
                    "predicted": label_names[pred],
                }
                if pred == true_label:
                    correct_examples.append(example)
                else:
                    incorrect_examples.append(example)

    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)

    return avg_loss, accuracy, all_labels, all_preds, correct_examples, incorrect_examples

# Perform evaluation and extract examples
test_loss, test_acc, true_labels, predictions, correct_examples, incorrect_examples = evaluate_with_examples(
    test_loader, classification_model, device, test_data, train_data.features["label"].names
)

# Classification Report
print("Error Analysis for Decoder-Only Model:")
print("Classification Report:")
print(classification_report(true_labels, predictions, target_names=train_data.features["label"].names))

# Randomly sample and display examples
print("\nRandom Sample of Incorrect Predictions for Decoder-Only:")
random_incorrect = random.sample(incorrect_examples, min(len(incorrect_examples), 5)) if incorrect_examples else []
for example in random_incorrect:
    print(example)

print("\nRandom Sample of Correct Predictions for Decoder-Only:")
random_correct = random.sample(correct_examples, min(len(correct_examples), 5)) if correct_examples else []
for example in random_correct:
    print(example)


Error Analysis for Encoder-Only Model:


Classification Report:
                                                  precision    recall  f1-score   support

                                activate_my_card       0.97      0.93      0.95        40
                                       age_limit       1.00      1.00      1.00        40
                         apple_pay_or_google_pay       0.98      1.00      0.99        40
                                     atm_support       0.97      0.97      0.97        40
                                automatic_top_up       1.00      0.90      0.95        40
         balance_not_updated_after_bank_transfer       0.70      0.75      0.72        40
balance_not_updated_after_cheque_or_cash_deposit       0.93      0.93      0.93        40
                         beneficiary_not_allowed       0.94      0.80      0.86        40
                                 cancel_transfer       0.97      0.95      0.96        40
                            card_about_to_expire       0.93      1.00      0

Evaluating: 100%|██████████| 193/193 [00:11<00:00, 16.20it/s]

Error Analysis for Decoder-Only Model:
Classification Report:
                                                  precision    recall  f1-score   support

                                activate_my_card       0.97      0.90      0.94        40
                                       age_limit       0.83      0.97      0.90        40
                         apple_pay_or_google_pay       1.00      1.00      1.00        40
                                     atm_support       0.89      0.97      0.93        40
                                automatic_top_up       0.94      0.85      0.89        40
         balance_not_updated_after_bank_transfer       0.50      0.80      0.62        40
balance_not_updated_after_cheque_or_cash_deposit       0.56      1.00      0.71        40
                         beneficiary_not_allowed       0.78      0.72      0.75        40
                                 cancel_transfer       1.00      0.85      0.92        40
                            card_abou

## Implementing My Suggestions to Improve Decoder-only Model

 I was successfully able to improve the accuracy of the decoder model by 12%, from 77% to 89%.

 These enhancements I suggested for the decoder-only model included a custom data augmentation strategy, prompt-based fine-tuning, and using a better optimizer. The primary aim was to address challenges in handling the categories that performed poorly, underrepresented patterns, and intents that have words in common while taking advantage GPT-2’s natural strengths in text generation and contextual understanding. The results of these improvements showed a notable increase in performance metrics, including accuracy, precision, recall, and F1-score, with the model achieving an overall accuracy of 89.4% and a weighted F1-score of 0.894.

The custom data augmentation strategy was critical in addressing underperforming categories. Additional synthetic examples were generated for classes such as "contactless_not_working" and "virtual_card_not_working," which previously exhibited low recall due to a lack of diverse training data. By exposing the model to varied examples of edge cases, the augmentation reduced overfitting on frequent patterns and improved generalization. A structured prompt, such as "Determine the intent of the following text: ...", allowed the model to better contextualize queries, leveraging its text completion capabilities to infer intent effectively. This was further supported by a tailored training configuration using the AdamW optimizer and a linear learning rate scheduler, which ensured smoother convergence and minimized overfitting.

The improvements were evident in the evaluation metrics. Categories with clear and distinct textual patterns, such as "activate_my_card" (F1: 0.99), "edit_personal_details" (F1: 0.99), and "verify_source_of_funds" (F1: 1.00), achieved near-perfect precision and recall. Additionally, previously challenging categories such as "virtual_card_not_working" (F1: 0.89) and "contactless_not_working" (F1: 0.91) showed significant improvement, demonstrating the effectiveness of the prompt-based approach and data augmentation. However, the model still faced challenges with overlapping categories and vague queries. For example, "why_verify_identity" (F1: 0.67) and "topping_up_by_card" (F1: 0.78) exhibited lower scores due to ambiguities in user inputs and insufficient distinguishing features in the text.

In [ ]:
def format_prompt(text):
    return f"Determine the intent of the following text:\n{text}\nReply with the intent label enclosed in <label_start> and <label_end>."

# Metric Function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), dim=-1).numpy()
    acc = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="weighted")
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

# Dataset Class
class IntentDataset(Dataset):
    def __init__(self, data, tokenizer, max_length):
        self.texts = data["text"]
        self.labels = data["label"]
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        prompt = format_prompt(text)
        encoded = self.tokenizer(
            prompt,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": encoded["input_ids"].squeeze(),
            "attention_mask": encoded["attention_mask"].squeeze(),
            "labels": label,
        }

# Load Model and Tokenizer
print("Loading GPT-2 tokenizer and model...")
model_name = "gpt2"
gpt2_tokenizer = GPT2Tokenizer.from_pretrained(model_name)
gpt2_tokenizer.padding_side = "left"
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token  # Set padding token to EOS

num_classes = len(train_data.features["label"].names)  # Number of labels
config = GPT2Config.from_pretrained(model_name, num_labels=num_classes)
classification_model = GPT2ForSequenceClassification.from_pretrained(model_name, config=config)
classification_model.resize_token_embeddings(len(gpt2_tokenizer))
classification_model.config.pad_token_id = classification_model.config.eos_token_id

# Dataset and Tokenization
print("Preparing datasets...")
seq_max_length = 128
train_dataset = IntentDataset(train_data, gpt2_tokenizer, seq_max_length)
test_dataset = IntentDataset(test_data, gpt2_tokenizer, seq_max_length)

# Custom Optimizer and Scheduler
print("Setting up optimizer and scheduler...")
optimizer = torch.optim.AdamW(classification_model.parameters(), lr=5e-5, eps=1e-8)

# Calculate total training steps
num_epochs = 3
batch_size = 8
num_training_steps = (len(train_dataset) // batch_size) * num_epochs

scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

# TrainingArguments
training_args = TrainingArguments(
    output_dir="output",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    learning_rate=5e-5,
    num_train_epochs=num_epochs,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",  # Set to "wandb", "tensorboard", etc. for integrations
)

# Trainer with Custom Optimizer and Scheduler
trainerbonus = Trainer(
    model=classification_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=gpt2_tokenizer,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, scheduler),  # Pass custom optimizer and scheduler
)

# Training
print("Starting training...")
trainerbonus.train()

# Evaluation
print("Evaluating on the test set...")
metrics = trainerbonus.evaluate()
print(metrics)

# Generate Predictions and Classification Report
print("Generating predictions for the test set...")
test_predictions = trainerbonus.predict(test_dataset)
predicted_labels = torch.argmax(torch.tensor(test_predictions.predictions), dim=-1).numpy()

# Print Classification Report
true_labels = test_predictions.label_ids
print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels, target_names=train_data.features["label"].names))

# Save Model
print("Saving the model...")
trainerbonus.save_model("final_model")


Loading GPT-2 tokenizer and model...


Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-23-1eda0c241aba>:95: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainerbonus = Trainer(


Preparing datasets...
Setting up optimizer and scheduler...
Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.998700,0.853604,0.759091,0.802876,0.759091,0.750759
2,0.488600,0.427603,0.880519,0.889401,0.880519,0.880199
3,0.254400,0.386035,0.894481,0.900416,0.894481,0.894422


Evaluating on the test set...


{'eval_loss': 0.38603490591049194, 'eval_accuracy': 0.8944805194805194, 'eval_precision': 0.9004164120071575, 'eval_recall': 0.8944805194805194, 'eval_f1': 0.8944216526213603, 'eval_runtime': 24.0422, 'eval_samples_per_second': 128.108, 'eval_steps_per_second': 16.014, 'epoch': 3.0}
Generating predictions for the test set...

Classification Report:
                                                  precision    recall  f1-score   support

                                activate_my_card       1.00      0.97      0.99        40
                                       age_limit       0.95      1.00      0.98        40
                         apple_pay_or_google_pay       0.95      1.00      0.98        40
                                     atm_support       0.93      1.00      0.96        40
                                automatic_top_up       1.00      0.88      0.93        40
         balance_not_updated_after_bank_transfer       0.74      0.70      0.72        40
balance_not_update